# Logistische regressie met _gradient descent_

In deze lab-sessie passen we logistische regressie met gradient descent toe op de **Iris dataset**. We bouwen een binaire classifier die onderscheid maakt tussen twee soorten irissen op basis van hun bloemeigenschappen.

## Leerdoelen
- Logistische regressie implementeren met gradient descent
- Binary cross-entropy loss berekenen en minimaliseren

## Extras
- Model performance evalueren
- Decision boundaries visualiseren

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

## 1. Data Loading en Exploratie

De Iris dataset bevat metingen van 150 iris bloemen van 3 verschillende soorten:
- **Setosa**
- **Versicolor** 
- **Virginica**

Voor elk bloem zijn er 4 features gemeten (in cm):
- Sepal length (kelkblad lengte)
- Sepal width (kelkblad breedte)
- Petal length (bloemblad lengte)
- Petal width (bloemblad breedte)

⚠️ Voor deze oefening maken we een **binaire classificatie**: we onderscheiden alleen tussen **Versicolor** (klasse 0) en **Virginica** (klasse 1).

In [ ]:
# Load the Iris dataset
iris = load_iris()

# Create a DataFrame for easier handling
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df["species"] = iris.target
df["species_name"] = df["species"].map({0: "setosa", 1: "versicolor", 2: "virginica"})

# Filter for binary classification: Versicolor (1) vs Virginica (2)
df_binary = df[df["species"].isin([1, 2])].copy()
# Remap to 0 and 1
df_binary["target"] = (df_binary["species"] == 2).astype(int)

print(f"Total samples: {len(df_binary)}")
print(f"Versicolor (0): {(df_binary['target'] == 0).sum()}")
print(f"Virginica (1): {(df_binary['target'] == 1).sum()}")
print(f"\nFeatures: {iris.feature_names}")
print("\nFirst few rows:")
df_binary.head(10)

In [ ]:
# Visualize the data - focusing on two features for visualization
# Create a scatter plot for petal dimensions
fig1 = px.scatter(
    df_binary,
    x="petal length (cm)",
    y="petal width (cm)",
    color="species_name",
    color_discrete_map={"versicolor": "#e377c2", "virginica": "#17becf"},
    title="Iris Dataset: Petal Dimensions",
    labels={"petal length (cm)": "Petal Length (cm)", "petal width (cm)": "Petal Width (cm)"},
    width=700,
    height=500,
)
fig1.update_traces(marker={"size": 10, "line": {"width": 1, "color": "white"}})
fig1.show()

# Create a scatter plot for sepal dimensions
fig2 = px.scatter(
    df_binary,
    x="sepal length (cm)",
    y="sepal width (cm)",
    color="species_name",
    color_discrete_map={"versicolor": "#e377c2", "virginica": "#17becf"},
    title="Iris Dataset: Sepal Dimensions",
    labels={"sepal length (cm)": "Sepal Length (cm)", "sepal width (cm)": "Sepal Width (cm)"},
    width=700,
    height=500,
)
fig2.update_traces(marker={"size": 10, "line": {"width": 1, "color": "white"}})
fig2.show()

## 2. Data Voorbereiding

We splitsen de data in:
- **Training set** (80%): gebruikt om het model te trainen
- **Test set** (20%): gebruikt om het model te evalueren

We gebruiken **feature scaling** (standaardisatie) om ervoor te zorgen dat alle features dezelfde schaal hebben. Dit helpt gradient descent sneller convergeren.

## 3. Logistische Regressie Implementatie

We implementeren logistische regressie met gradient descent zoals gezien in de cursus:

### Model
$$
\pmb{\hat{y}} = \sigma(\pmb{X}\pmb{b}) = \frac{1}{1 + e^{-\pmb{X}\pmb{b}}}
$$

### Binary Cross-Entropy Loss
$$
\mathcal{L}_{BCE}(\pmb{b}) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]
$$

### Gradiënt
$$
\nabla \mathcal{L}_{BCE} = \frac{1}{n} \pmb{X}^T(\pmb{\hat{y}} - \pmb{y})
$$

### Gradient Descent Update
$$
\pmb{b}^{k+1} = \pmb{b}^k - \lambda \nabla \mathcal{L}_{BCE}(\pmb{b}^k)
$$

## 4. Training met Gradient Descent

Nu trainen we het model met gradient descent. We houden de loss en accuracy bij voor zowel de training als test set.

## 5. Visualisatie van Training Proces

We visualiseren hoe de loss en accuracy evolueren tijdens het training proces.

## 6. Model Evaluatie en Predictions

Laten we kijken naar de voorspellingen van ons model en hoe goed het presteert.

In [ ]:
# Confusion Matrix visualization
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_test)

# Create confusion matrix heatmap using plotly
cm_df = pd.DataFrame(
    cm,
    index=["Versicolor (0)", "Virginica (1)"],
    columns=["Versicolor (0)", "Virginica (1)"],
)

fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Confusion Matrix",
    labels={"x": "Predicted Label", "y": "True Label", "color": "Count"},
    width=600,
    height=500,
)
fig.update_traces(textfont_size=20)
fig.show()

print("\nConfusion Matrix Analysis:")
print(f"True Negatives (Versicolor correctly classified): {cm[0, 0]}")
print(f"False Positives (Versicolor misclassified as Virginica): {cm[0, 1]}")
print(f"False Negatives (Virginica misclassified as Versicolor): {cm[1, 0]}")
print(f"True Positives (Virginica correctly classified): {cm[1, 1]}")

## 7. Decision Boundary Visualisatie

We visualiseren de decision boundary van ons model voor twee features tegelijk. We gebruiken petal length en petal width omdat deze features het best discrimineren tussen de twee soorten.

In [ ]:
# For visualization, we'll train a simple 2D model with just petal length and width
# This allows us to visualize the decision boundary

# Select only petal features (indices 2 and 3)
X_train_2d = X_train_scaled[:, [2, 3]]  # Petal length and width
X_test_2d = X_test_scaled[:, [2, 3]]

# Add intercept
X_train_2d_final = np.column_stack([np.ones(len(X_train_2d)), X_train_2d])
X_test_2d_final = np.column_stack([np.ones(len(X_test_2d)), X_test_2d])

# Train a 2D model
b_2d = np.zeros(3)
learning_rate_2d = 0.1
max_iterations_2d = 5000

for _ in range(max_iterations_2d):
    grad_2d = gradient_bce(X_train_2d_final, y_train, b_2d)
    b_2d = b_2d - learning_rate_2d * grad_2d

acc_2d = accuracy(X_test_2d_final, y_test, b_2d)
print(f"2D Model Test Accuracy: {acc_2d:.2%}")
print(f"2D Model Parameters: b₀={b_2d[0]:.3f}, b₁={b_2d[1]:.3f}, b₂={b_2d[2]:.3f}")

In [ ]:
# Create a mesh grid for decision boundary visualization
x1_min, x1_max = X_train_2d[:, 0].min() - 0.5, X_train_2d[:, 0].max() + 0.5
x2_min, x2_max = X_train_2d[:, 1].min() - 0.5, X_train_2d[:, 1].max() + 0.5

xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200), np.linspace(x2_min, x2_max, 200))

# Create feature matrix for grid
X_grid = np.column_stack([np.ones(xx1.ravel().shape[0]), xx1.ravel(), xx2.ravel()])

# Predict probabilities for each point in the grid
Z = sigmoid(X_grid @ b_2d)
Z = Z.reshape(xx1.shape)

# Create contour plot using plotly
fig = px.imshow(
    Z,
    x=np.linspace(x1_min, x1_max, 200),
    y=np.linspace(x2_min, x2_max, 200),
    color_continuous_scale="RdYlBu_r",
    origin="lower",
    title="Decision Boundary: Logistic Regression on Iris Dataset",
    labels={
        "x": "Petal Length (standardized)",
        "y": "Petal Width (standardized)",
        "color": "P(Virginica)",
    },
    width=900,
    height=700,
)
fig.update_coloraxes(showscale=False)

# Add training data points
train_df = pd.DataFrame(
    {
        "Petal Length": X_train_2d[:, 0],
        "Petal Width": X_train_2d[:, 1],
        "Species": ["Versicolor" if y == 0 else "Virginica" for y in y_train],
        "Type": "Training",
    }
)

# Add test data points
test_df = pd.DataFrame(
    {
        "Petal Length": X_test_2d[:, 0],
        "Petal Width": X_test_2d[:, 1],
        "Species": ["Versicolor" if y == 0 else "Virginica" for y in y_test],
        "Type": "Test",
    }
)

# Add scatter traces for training data
for species in ["Versicolor", "Virginica"]:
    train_species = train_df[train_df["Species"] == species]
    color = "#e377c2" if species == "Versicolor" else "#17becf"
    fig.add_scatter(
        x=train_species["Petal Length"],
        y=train_species["Petal Width"],
        mode="markers",
        marker={
            "size": 12,
            "color": color,
            "line": {"width": 2, "color": "white"},
            "symbol": "circle",
        },
        name=f"{species} (train)",
    )

# Add scatter traces for test data
for species in ["Versicolor", "Virginica"]:
    test_species = test_df[test_df["Species"] == species]
    color = "#e377c2" if species == "Versicolor" else "#17becf"
    fig.add_scatter(
        x=test_species["Petal Length"],
        y=test_species["Petal Width"],
        mode="markers",
        marker={
            "size": 12,
            "color": color,
            "line": {"width": 2, "color": "black"},
            "symbol": "triangle-up",
        },
        name=f"{species} (test)",
    )

fig.show()

print("\nDecision boundary equation (in standardized space):")
print(f"  {b_2d[0]:.3f} + {b_2d[1]:.3f}×(petal_length) + {b_2d[2]:.3f}×(petal_width) = 0")

## 8. Interactieve Visualisatie met Plotly

Laten we een interactieve 3D visualisatie maken om te zien hoe de voorspelde waarschijnlijkheid varieert in de feature space.

In [ ]:
# Create 3D surface plot with Plotly
# Prepare data for the surface
surface_df = pd.DataFrame(
    {
        "Petal Length": xx1.ravel(),
        "Petal Width": xx2.ravel(),
        "P(Virginica)": Z.ravel(),
    }
)

# Create 3D scatter plot for training data
train_3d_df = pd.DataFrame(
    {
        "Petal Length": X_train_2d[:, 0],
        "Petal Width": X_train_2d[:, 1],
        "True Class": y_train,
        "Species": ["Versicolor" if y == 0 else "Virginica" for y in y_train],
        "Type": "Training",
    }
)

# Create 3D scatter plot for test data
test_3d_df = pd.DataFrame(
    {
        "Petal Length": X_test_2d[:, 0],
        "Petal Width": X_test_2d[:, 1],
        "True Class": y_test,
        "Species": ["Versicolor" if y == 0 else "Virginica" for y in y_test],
        "Type": "Test",
    }
)

# Start with 3D scatter for training data
fig = px.scatter_3d(
    train_3d_df,
    x="Petal Length",
    y="Petal Width",
    z="True Class",
    color="Species",
    color_discrete_map={"Versicolor": "#e377c2", "Virginica": "#17becf"},
    symbol="Type",
    title="3D Visualization: Logistic Regression Probability Surface",
    width=900,
    height=700,
)

# Add test data
fig.add_scatter3d(
    x=test_3d_df["Petal Length"],
    y=test_3d_df["Petal Width"],
    z=test_3d_df["True Class"],
    mode="markers",
    marker={
        "size": 8,
        "color": ["#e377c2" if s == "Versicolor" else "#17becf" for s in test_3d_df["Species"]],
        "symbol": "diamond",
        "line": {"color": "black", "width": 2},
    },
    name="Test Data",
)

# For the 3D surface, we need plotly.graph_objects (een uitbreiding van plotly.express)
fig.add_trace(
    go.Surface(
        x=np.linspace(x1_min, x1_max, 200),
        y=np.linspace(x2_min, x2_max, 200),
        z=Z,
        colorscale="RdYlBu_r",
        opacity=0.7,
        name="P(Virginica)",
        showscale=True,
        colorbar={"title": "P(Virginica)", "x": 1.1},
    )
)

# Update layout
fig.update_layout(
    scene={
        "xaxis_title": "Petal Length (std)",
        "yaxis_title": "Petal Width (std)",
        "zaxis_title": "Probability / True Class",
        "camera": {"eye": {"x": 1.5, "y": -1.5, "z": 1.2}},
    }
)

fig.show()